In [9]:

import sys
import logging
sys.path.append('E:\wangzhilin\QuantumGalaxy')
from QGI.mysql import MYSQL

import pandas as pd
import numpy as np
from QGI.neoapi import Neo4j

In [10]:
def get_logger():
    logger=logging.getLogger('logger')
    logger.setLevel(logging.INFO)
    #fh=logging.FileHandler("E:/wangzhilin/QuantumGalaxy/logs/aura_api_log.log",'a',encoding='utf-8')
    #fh.setLevel(logging.INFO)
    ch=logging.StreamHandler()
    ch.setLevel(logging.DEBUG)
    formatter=logging.Formatter(fmt='%(asctime)s %(name)-12s %(levelname)-8s %(message)s',datefmt='%m-%d %H:%M')
    ch.setFormatter(formatter)
    #fh.setFormatter(formatter)
    logger.addHandler(ch)
    #logger.addHandler(fh)
    return logger
logger=logging.getLogger()

In [12]:
uri = "neo4j+ssc://08ef0a79.databases.neo4j.io"
user = "QG_Editor"
password = "editor"


neo=Neo4j(uri,user,password)


In [13]:

host='localhost'
user='Local_Editor'
password='QuantumGalaxy'
database='qgdbs'
mysql=MYSQL(host,user,password,database)

# 写入单个record

In [46]:
field='name'


In [47]:
node_id=mysql.read_query('select id from nodes where name like"%兰花科创%"')[0][0]
node_id

9

In [59]:
def get_record_index(node_id,field,main=False):
    if main:
        return 0
    sql='select count(*) from nodes_records where id=%s and field like "%%%s%%" and not record_index=0'%(node_id,field)
    #print(sql)
    r=mysql.read_query(sql)[0][0]+1
    
    return r


In [127]:
record_index=get_record_index(node_id,'disadvantage')

record_index

1

In [128]:
val=[]

In [129]:
idx=[node_id,record_index,'disadvantage']
data=['产能差距：兰花科创无烟煤核定产能900万吨每年， ；华阳股份,核定在产产能3270 万吨,权益产能2836万吨(占比 87%),其中大部分为稀缺煤种无烟煤。2021年中国尿素产能第一为山西天泽（290万吨，占比4.03%），第四为华鲁恒升（155万吨，占比2.15%），兰花科创为年尿素产能 100 万吨。己内酰胺：截止2021年7月，兰花科创拥有当时国内3%的产能，低于大部分同行企业产能，产能市占率较低，产能市占率第一为福建申远占13%。',None,None,None,None,'杨丹',None,None,None]
#data=['营销研发费用为1.6亿元',160000000,'2021-01-01','2021-12-31','2021年报','杨丹',None,None,'RMB']
#conclusion,value,start_date,end_date,source,writer,demonstration,region,currency
metaval=idx+data

metaval=tuple(v for v in metaval)
val.append(metaval)
val

[(9,
  1,
  'disadvantage',
  '产能差距：兰花科创无烟煤核定产能900万吨每年， ；华阳股份,核定在产产能3270 万吨,权益产能2836万吨(占比 87%),其中大部分为稀缺煤种无烟煤。2021年中国尿素产能第一为山西天泽（290万吨，占比4.03%），第四为华鲁恒升（155万吨，占比2.15%），兰花科创为年尿素产能 100 万吨。己内酰胺：截止2021年7月，兰花科创拥有当时国内3%的产能，低于大部分同行企业产能，产能市占率较低，产能市占率第一为福建申远占13%。',
  None,
  None,
  None,
  None,
  '杨丹',
  None,
  None,
  None)]

In [130]:

sql='insert into nodes_records (id,record_index,field,conclusion,value,start_date,end_date,source,writer,demonstration,region,currency) values (%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s) '
print(sql)

insert into nodes_records (id,record_index,field,conclusion,value,start_date,end_date,source,writer,demonstration,region,currency) values (%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s) 


In [131]:
mysql.write_many_query(sql,val)

1

# 从mysql写neo节点中字段

In [5]:
name='测试产品'
label='product'
sql='select * from nodes where name like "%%%s%%" and labels like "%%%s%%"'%(name,label)
id=mysql.read_query(sql)
if len(id)!=1:
    
    raise Exception('find %d nodes'%len(id))
id=id[0][0]
#id
sql='select field,conclusion from nodes_records where id=%s and record_index=0'%id
records=mysql.read_query(sql)
records

(('advantage', '没有优点'),
 ('disadvantage', '需要双端同步，需要两头写入'),
 ('name', '测试产品'),
 ('rd_cost', '3个人两星期'))

In [6]:
cypher='match (n) where id(n)=%s set'%id
#neo.read_query(cypher)

In [7]:
for r in records:
    cypher+=' n.%s="%s",'%(r[0],r[1])
cypher=cypher[:-1]
cypher

'match (n) where id(n)=13 set n.advantage="没有优点", n.disadvantage="需要双端同步，需要两头写入", n.name="测试产品", n.rd_cost="3个人两星期"'

In [8]:
neo.simple_query(cypher)

[]